# KV Cache — Runnable Code

Companion notebook for the article: **KV Cache: Intuition, Implementation, Production**

Three levels:
1. **The Problem** — standard attention and why generation gets slower every step
2. **The Fix** — add a cache with 3 lines, benchmark the speedup
3. **Production** — INT8 quantization, Grouped Query Attention (GQA)

No GPU needed. Runs on CPU in under 10 seconds.

> The `Head` class here is real transformer code — the same pattern used in GPT-2 and nanoGPT.
> When the attention-mechanism article is published, this class will be covered in full detail.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.11.0


## Level 1 — The Problem

Every time you generate a new token, a transformer runs attention over the **full sequence so far**.
For a sequence of length T, that's T key vectors, T value vectors, and a T×T attention matrix.
Generate 200 tokens and you've run attention 200 times — each time over a longer sequence.

Here's what standard single-head attention looks like. Watch where K and V are computed.

In [2]:
class Head(nn.Module):
    """Single-head causal self-attention — standard, no caching."""

    def __init__(self, n_embed, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embed,head_size,bias=False)
        self.query = nn.Linear(n_embed,head_size,bias=False)
        self.value = nn.Linear(n_embed,head_size,bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size) — computed for ALL T tokens
        q = self.query(x)  # (B, T, head_size) — computed for ALL T tokens
        v = self.value(x)  # (B, T, head_size) — computed for ALL T tokens

        wei = q @ k.transpose(-2, -1) * C**(-0.5) #(B,T,head_size) @ (B,head_size,T) --> (B,T,T) 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))#(B,T,T) 
        wei = F.softmax(wei, dim=-1)#(B,T,T) 
        wei = self.dropout(wei)
        return wei @ v #(B,T,T) @ (B,T,head_size) --> (B,T,C) # (B, T, head_size)

In [3]:
@torch.no_grad()
def generate_no_cache(head, n_embed, prompt_len=10, n_new_tokens=10):
    """Simulate token generation without a cache.
    At every step: feed the full growing sequence through attention."""
    lm_head = nn.Linear(n_embed, n_embed)
    x = torch.randn(1, prompt_len, n_embed)  # simulated prompt

    for _ in range(n_new_tokens):
        out = head(x)                                 # recomputes K,V for ALL past tokens
        next_tok = lm_head(out[:, -1:, :])            # predict next token from last position
        x = torch.cat([x, next_tok], dim=1)           # sequence grows by 1

    return x.shape[1]  # final sequence length

n_embed, head_size, block_size = 64, 64, 512
head = Head(n_embed, head_size, block_size)
final_len = generate_no_cache(head, n_embed, prompt_len=10, n_new_tokens=10)
print(f"Final sequence length: {final_len}")  # 20

Final sequence length: 20


In [5]:
print("Time per attention call as sequence length grows (no cache):\n")
print(f"  {'seq_len':>8}  {'avg ms':>10}")
print("  " + "-" * 22)

for seq_len in [10, 50, 100, 200, 400]:
    times = []
    for _ in range(30):
        x = torch.randn(1, seq_len, n_embed)
        t0 = time.perf_counter()
        with torch.no_grad():
            head(x)
        times.append(time.perf_counter() - t0)
    avg_ms = sum(times) / len(times) * 1000
    print(f"  {seq_len:8d}  {avg_ms:8.3f}ms")

print("\nEach decode step is slower than the last — O(n²) attention over a growing sequence.")

Time per attention call as sequence length grows (no cache):

   seq_len      avg ms
  ----------------------
        10     0.174ms
        50     0.076ms
       100     0.157ms
       200     0.304ms
       400     0.488ms


RuntimeError: The size of tensor a (512) must match the size of tensor b (800) at non-singleton dimension 2

## Level 2 — The Fix

The problem: K and V are recomputed for every past token at every step.
The fix: **compute K and V once per token, cache them, reuse**.

Q is never cached — it's only needed for the current token making the decision.

The change to `Head.forward()` is three lines. Everything else stays identical.

In [5]:
class HeadWithCache(nn.Module):
    """Same as Head — with KV cache added in 3 lines (marked ← NEW)."""

    def __init__(self, n_embed, head_size, block_size, dropout=0.0):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, past_kv=None):              # ← NEW: accept cached K, V
        B, T, C = x.shape
        k = self.key(x)    # only for the NEW token(s) in x
        q = self.query(x)
        v = self.value(x)

        if past_kv is not None:                       # ← NEW: prepend past K, V
            k = torch.cat([past_kv[0], k], dim=1)
            v = torch.cat([past_kv[1], v], dim=1)

        T_full = k.shape[1]
        wei = q @ k.transpose(-2, -1) * C**(-0.5)
        if T > 1:  # prefill: causal mask applies; decode (T=1): new token sees all past
            wei = wei.masked_fill(self.tril[:T, :T_full] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v

        return out, (k, v)                            # ← NEW: return updated cache

In [6]:
@torch.no_grad()
def generate_with_cache(head, n_embed, prompt_len=10, n_new_tokens=10):
    """Generate tokens with KV cache.
    Prefill: process full prompt once.
    Decode: feed only the new token each step."""
    lm_head = nn.Linear(n_embed, n_embed)
    prompt = torch.randn(1, prompt_len, n_embed)

    # Prefill — process the whole prompt, build the initial cache
    _, past_kv = head(prompt, past_kv=None)
    print(f"After prefill  — cache K shape: {past_kv[0].shape}")

    for step in range(n_new_tokens):
        new_tok = torch.randn(1, 1, n_embed)         # only 1 token!
        out, past_kv = head(new_tok, past_kv=past_kv)
        if step < 3:
            print(f"After step {step+1:2d}  — cache K shape: {past_kv[0].shape}")

    print(f"...")
    print(f"Final cache K shape: {past_kv[0].shape}")
    return past_kv[0].shape[1]

head_cache = HeadWithCache(n_embed, head_size, block_size)
generate_with_cache(head_cache, n_embed, prompt_len=10, n_new_tokens=10)

After prefill  — cache K shape: torch.Size([1, 10, 64])
After step  1  — cache K shape: torch.Size([1, 11, 64])
After step  2  — cache K shape: torch.Size([1, 12, 64])
After step  3  — cache K shape: torch.Size([1, 13, 64])
...
Final cache K shape: torch.Size([1, 20, 64])


20

In [7]:
print("Benchmark — cached vs no cache:\n")
print(f"  {'seq_len':>8}  {'no cache':>12}  {'with cache':>12}  {'speedup':>10}")
print("  " + "-" * 48)

head_base  = Head(n_embed, head_size, block_size)
head_cache = HeadWithCache(n_embed, head_size, block_size)

for seq_len in [50, 100, 200, 500]:

    # No cache: re-run full growing sequence at every decode step
    t0 = time.perf_counter()
    with torch.no_grad():
        for i in range(seq_len):
            x = torch.randn(1, i + 1, n_embed)
            head_base(x)
    no_cache_s = time.perf_counter() - t0

    # With cache: prefill once, then single-token steps
    t0 = time.perf_counter()
    with torch.no_grad():
        prompt = torch.randn(1, 1, n_embed)
        _, past_kv = head_cache(prompt)
        for _ in range(seq_len - 1):
            tok = torch.randn(1, 1, n_embed)
            _, past_kv = head_cache(tok, past_kv=past_kv)
    cache_s = time.perf_counter() - t0

    print(f"  {seq_len:8d}  {no_cache_s:10.3f}s  {cache_s:10.3f}s  {no_cache_s/cache_s:8.1f}x")

print("\nSpeedup grows with sequence length — no-cache is O(n²), cached is O(n).")

Benchmark — cached vs no cache:

   seq_len      no cache    with cache     speedup
  ------------------------------------------------
        50       0.004s       0.002s       2.1x
       100       0.009s       0.003s       2.9x
       200       0.028s       0.007s       3.8x


       500       0.156s       0.019s       8.2x

Speedup grows with sequence length — no-cache is O(n²), cached is O(n).


## Level 3 — Production Extensions

Two techniques production systems layer on top of KV cache:

| Technique | What it does | Saving |
|-----------|-------------|--------|
| INT8 quantization | Store K/V in 8-bit instead of 32-bit | 4× memory |
| Grouped Query Attention | Multiple Q heads share one K/V head | 4–8× KV cache size |

These are additive — real deployments use both.

### INT8 Quantization

A 70B model serving 1000 tokens generates ~2.5 GB of KV cache in FP32.
INT8 cuts that to ~625 MB. The precision loss is small enough to ignore in practice.

In [8]:
def quantize_cache(tensor: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    scale = tensor.abs().max() / 127
    quantized = (tensor / scale).round().clamp(-128, 127).to(torch.int8)
    return quantized, scale


def dequantize_cache(quantized: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return quantized.float() * scale


# 70B model, 1000 tokens, d_model=8192, but scaled down to d_model=512 for demo
cache_fp32 = torch.randn(1, 1000, 512)
cache_int8, scale = quantize_cache(cache_fp32)
cache_back  = dequantize_cache(cache_int8, scale)

fp32_mb = cache_fp32.nelement() * cache_fp32.element_size() / 1e6
int8_mb  = cache_int8.nelement()  * cache_int8.element_size()  / 1e6
max_err  = (cache_fp32 - cache_back).abs().max().item()

print(f"FP32 cache: {fp32_mb:.2f} MB")
print(f"INT8 cache: {int8_mb:.2f} MB")
print(f"Compression: {fp32_mb / int8_mb:.0f}x")
print(f"Max absolute error: {max_err:.5f}")

FP32 cache: 2.05 MB
INT8 cache: 0.51 MB
Compression: 4x
Max absolute error: 0.01836


### Grouped Query Attention (GQA)

Standard MHA: each of the N query heads has its own K and V head → N KV heads total.
GQA: every group of query heads shares one K/V head.

Llama 2 70B: 64 Q heads, 8 KV heads → 8× smaller KV cache at the architecture level.
`group_size = n_q_heads / n_kv_heads = 64 / 8 = 8`

Q, K, V are packed as `(batch * n_heads, seq_len, d_head)` — heads live in the batch dimension.
Expanding K/V to match Q means repeating along `dim=0` (the head dimension), not `dim=1` (sequence).

In [9]:
def standard_mha(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    scale = math.sqrt(q.shape[-1])
    scores = torch.bmm(q, k.transpose(1, 2)) / scale
    return torch.bmm(F.softmax(scores, dim=-1), v)


def grouped_query_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    n_q_heads: int,
    n_kv_heads: int,
) -> torch.Tensor:
    group_size = n_q_heads // n_kv_heads
    k = k.repeat_interleave(group_size, dim=0)   # expand heads, not sequence positions
    v = v.repeat_interleave(group_size, dim=0)
    return standard_mha(q, k, v)


batch, seq_len, d_head = 1, 20, 64
n_q_heads, n_kv_heads  = 8, 2   # 4:1 ratio

q = torch.randn(batch * n_q_heads,  seq_len, d_head)   # (8, 20, 64)
k = torch.randn(batch * n_kv_heads, seq_len, d_head)   # (2, 20, 64)
v = torch.randn(batch * n_kv_heads, seq_len, d_head)   # (2, 20, 64)

out = grouped_query_attention(q, k, v, n_q_heads, n_kv_heads)
print(f"GQA output shape: {out.shape}")                # (8, 20, 64)

mha_kv = 2 * n_q_heads  * seq_len * d_head
gqa_kv = 2 * n_kv_heads * seq_len * d_head
print(f"\nMHA KV cache elements: {mha_kv:,}")
print(f"GQA KV cache elements: {gqa_kv:,}")
print(f"Reduction: {mha_kv // gqa_kv}×")

GQA output shape: torch.Size([8, 20, 64])

MHA KV cache elements: 20,480
GQA KV cache elements: 5,120
Reduction: 4×


## Summary

| Technique | What it changes | Typical gain |
|-----------|----------------|--------------|
| KV Cache | Skip K/V recompute for past tokens | 5–15× latency |
| INT8 Quantization | Store K/V in 8-bit | 4× memory |
| Grouped Query Attention | Share K/V heads across query groups | 4–8× KV cache size |

These are additive. A production deployment (e.g. vLLM + Llama 3) uses all three.

**Read next:** [Autoregressive Decoding](#) — how the generate loop works end to end.